In [ ]:
from asapdiscovery.data.readers.molfile import MolFileFactory
import pandas as pd
import argparse
from pathlib import Path
import json
import numpy as np

In [ ]:
ligs = MolFileFactory(filename="/data1/choderaj/paynea/asap-datasets/full_cross_dock/ligand_files/combined_3d.sdf").load()

In [ ]:
compound_name = [lig.compound_name for lig in ligs]
xtal_name = [lig.tags["xtal_name"] for lig in ligs]
df1 = pd.DataFrame({"compound_name": compound_name, "xtal_name": xtal_name})

In [ ]:
with open("/data1/choderaj/paynea/asap-datasets/full_cross_dock/cmpd_date_dict/date_dict.json", "r") as f:
    date_dict = json.load(f)

In [ ]:
df1["date"] = df1.xtal_name.apply(lambda x: date_dict.get(x.split("_")[0], np.nan))

In [ ]:
df1.nunique()

In [ ]:
grouped = df1.groupby("compound_name").count()[["xtal_name"]]

In [ ]:
dups = grouped[grouped["xtal_name"] > 1].index

In [ ]:
dups

In [ ]:
dict_data = [{"smiles": lig.smiles, 
              "compound_name": lig.compound_name, 
              "series": lig.tags['xtal_name'][5], 
              "number": lig.tags['xtal_name'].split("_")[0][6:], 
              "xtal_id": lig.tags['xtal_name'].split("_")[1], 
              "xtal_name": lig.tags['xtal_name'], 
              "lig": lig,
              "structure_name": lig.tags['xtal_name'][:-3]} for lig in ligs]
df = pd.DataFrame.from_records(dict_data)
df = df[df["series"].isin(["x", "P"])]
# unique_compounds = df.sort_values(["series", "number", "compound_name"], ascending=[False, False, False]).groupby("compound_name").head(1).groupby("smiles").head(1)
unique_compounds = df.sort_values(["series", "number", "compound_name"], ascending=[False, False, False]).groupby("smiles").head(1)
unique_compounds.groupby("series").count()

In [ ]:
unique_compounds.nunique()

In [ ]:
grouped = unique_compounds.groupby("compound_name").count()[["smiles"]]

In [ ]:
compound_name_with_multiple_smiles = grouped[grouped["smiles"] > 1].index

In [ ]:
compound_name_with_multiple_smiles

In [ ]:
unique_compounds[unique_compounds['compound_name'].isin(compound_name_with_multiple_smiles)]

In [ ]:
for lig in unique_compounds[unique_compounds['compound_name'].isin(compound_name_with_multiple_smiles)]['lig']:
    lig.to_sdf(f"{lig.compound_name}_{lig.tags['xtal_name']}.sdf")

In [ ]:
df1[df1["compound_name"].isin(compound_name_with_multiple_smiles)]

In [ ]:
set(unique_compounds.xtal_name).difference(set(df1.xtal_name))

In [ ]:
set(unique_compounds.xtal_name.unique()) - set(df1.xtal_name.unique())

In [ ]:
removed = set(df1.xtal_name.unique()) - set(unique_compounds.xtal_name.unique())

In [ ]:
og_removed = df1[df1["xtal_name"].isin(removed)]

In [ ]:
new_removed = df1[df1["compound_name"].isin(dups)]

In [ ]:
len(og_removed)

In [ ]:
len(new_removed)